# 🚀 CIFAR-100 Training on Colab
自動掛載 Google Drive、clone repo、偵測 checkpoint 並恢復訓練。

## Step 1｜掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Checkpoint 目錄：{CHECKPOINT_DIR}')

!git config --global user.name "rvgoing"
!git config --global user.email "cmkdkimo@gmail.com"

## Step 2｜Clone / Pull 最新程式碼

In [ ]:
GITHUB_REPO = 'https://github.com/rvgoing/CFIR_100_ReFresh.git'
REPO_NAME   = 'CFIR_100_ReFresh'

if os.path.exists(f'/content/{REPO_NAME}'):
    print('📦 Repo 已存在，執行 git pull ...')
    %cd /content/{REPO_NAME}
    !git pull
else:
    print('📦 Clone repo ...')
    !git clone {GITHUB_REPO}
    %cd /content/{REPO_NAME}

print('✅ 程式碼已是最新版本')

## Step 3｜安裝依賴

In [ ]:
!pip install -q -r requirements.txt
print('✅ 套件安裝完成')

## Step 4｜確認 GPU

In [ ]:
import torch
print(f'PyTorch 版本：{torch.__version__}')
print(f'CUDA 可用：{torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU：{torch.cuda.get_device_name(0)}')
    print(f'VRAM：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
!nvidia-smi

## Step 5｜自動偵測 Checkpoint 並開始訓練

In [ ]:
import os

# 注意：副檔名是 .pth
BEST_CKPT = os.path.join(CHECKPOINT_DIR, 'model_best.pth')
LAST_CKPT = os.path.join(CHECKPOINT_DIR, 'checkpoint.pth')

if os.path.exists(LAST_CKPT):
    resume_path = LAST_CKPT
    print(f'🔄 發現 checkpoint，將從上次進度恢復：{resume_path}')
elif os.path.exists(BEST_CKPT):
    resume_path = BEST_CKPT
    print(f'🔄 發現 best checkpoint，將從此恢復：{resume_path}')
else:
    resume_path = ''
    print('🆕 未發現 checkpoint，從頭開始訓練')

resume_arg = f'--resume {resume_path}' if resume_path else ''

cmd = f"python train.py --epochs 20 --batch-size 128 --lr 0.1 --save-dir {CHECKPOINT_DIR} {resume_arg}"

print(f'\n🚀 執行指令：\n{cmd}\n')
!{cmd}

## Step 6｜確認 Checkpoint 已存到 Drive

In [ ]:
import os

print('📂 Drive checkpoint 目錄內容：')
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    size = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1024**2
    print(f'  {f}  ({size:.1f} MB)')

## Step 7｜執行評量（訓練完成後執行）

In [ ]:
EVAL_DIR = os.path.join(CHECKPOINT_DIR, 'eval_results')
os.makedirs(EVAL_DIR, exist_ok=True)

cmd = f"python evaluate.py --checkpoint {BEST_CKPT} --save-dir {EVAL_DIR}"

print(f'🔍 執行評量：\n{cmd}\n')
!{cmd}

print('\n📂 評量結果：')
for f in sorted(os.listdir(EVAL_DIR)):
    size = os.path.getsize(os.path.join(EVAL_DIR, f)) / 1024**2
    print(f'  {f}  ({size:.1f} MB)')